# Comparaison des Techniques de Traitement du Déséquilibre

Ce notebook compare systématiquement les techniques de gestion du déséquilibre de classes
pour la détection de fraude :
- **Sans rééchantillonnage** (baseline)
- **SMOTE** (Synthetic Minority Over-sampling Technique)
- **ADASYN** (Adaptive Synthetic Sampling)
- **Random Undersampling** (sous-échantillonnage aléatoire)
- **SMOTE + Tomek Links** (combinaison sur/sous-échantillonnage)
- **Cost-Sensitive Learning** (pondération des classes)

Chaque technique est évaluée avec un classifieur **XGBoost** sur le jeu de test.

In [ ]:
import sys
import os
import warnings
import time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

# Ajouter le répertoire racine au path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.balancer import ImbalanceHandler
from src.data.loader import load_creditcard
from src.data.preprocessor import FraudPreprocessor
from src.data.splitter import stratified_split
from src.evaluation.metrics import compute_all_metrics
from config import (
    FIGURES_DIR, FIGURE_DPI, DATA_PROCESSED_DIR,
    XGBOOST_PARAMS, TARGET_COL, RANDOM_STATE
)

# Style graphique
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = FIGURE_DPI

# Répertoire de sauvegarde
COMP_DIR = os.path.join(FIGURES_DIR, 'comparison')
os.makedirs(COMP_DIR, exist_ok=True)

print('Imports chargés avec succès.')

In [ ]:
# Charger les données prétraitées (ou prétraiter si non disponibles)
try:
    X_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_train.csv'))
    X_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_test.csv'))
    y_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_train.csv')).squeeze()
    y_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_test.csv')).squeeze()
    print(f'Données prétraitées chargées depuis {DATA_PROCESSED_DIR}')

except FileNotFoundError:
    print('Données prétraitées non trouvées. Exécution du pipeline de prétraitement...')
    df = load_creditcard()
    preprocessor = FraudPreprocessor(scaler_type='standard')
    df = preprocessor.handle_missing(df)
    df = preprocessor.engineer_features_kaggle(df)

    feature_cols = [c for c in df.columns if c != TARGET_COL]
    X = df[feature_cols]
    y = df[TARGET_COL]

    cols_to_scale = ['Amount', 'Time', 'Hour', 'Amount_Log', 'V_Mean', 'V_Std']
    X_scaled = preprocessor.fit_transform(X, columns=cols_to_scale)

    X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
        X_scaled, y, test_size=0.05, val_size=0.15
    )
    print('Pipeline de prétraitement exécuté avec succès.')

print(f'\nTrain : {X_train.shape[0]:,} samples ({y_train.sum():,.0f} fraudes, {y_train.mean()*100:.3f}%)')
print(f'Test  : {X_test.shape[0]:,} samples ({y_test.sum():,.0f} fraudes, {y_test.mean()*100:.3f}%)')

## Distribution Originale

Avant d'appliquer les techniques de rééchantillonnage, examinons la distribution
des classes dans l'ensemble d'entraînement.

In [ ]:
# Distribution originale dans l'ensemble d'entraînement
n_legit = int((y_train == 0).sum())
n_fraud = int((y_train == 1).sum())
total = len(y_train)

print('Distribution des classes (ensemble d\'entraînement) :')
print(f'  Légitime (0) : {n_legit:>10,}  ({n_legit/total*100:.3f}%)')
print(f'  Fraude   (1) : {n_fraud:>10,}  ({n_fraud/total*100:.3f}%)')
print(f'  Total        : {total:>10,}')
print(f'  Ratio déséquilibre : {n_legit/max(n_fraud,1):.0f}:1')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Légitime (0)', 'Fraude (1)'], [n_legit, n_fraud],
              color=colors, edgecolor='black')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution Originale — Ensemble d\'Entraînement', fontsize=14, fontweight='bold')
for bar, count in zip(bars, [n_legit, n_fraud]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{count:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## Application des Techniques de Rééchantillonnage

Nous appliquons les 4 techniques de rééchantillonnage implémentées dans `ImbalanceHandler` :

| Technique | Type | Principe |
|-----------|------|----------|
| **SMOTE** | Sur-échantillonnage | Génère des exemples synthétiques par interpolation k-NN |
| **ADASYN** | Sur-échantillonnage adaptatif | Génère plus d'exemples dans les zones difficiles |
| **Random Undersampling** | Sous-échantillonnage | Supprime aléatoirement des exemples majoritaires |
| **SMOTE + Tomek** | Hybride | SMOTE puis nettoyage des liens Tomek à la frontière |

In [ ]:
# Application des techniques de rééchantillonnage
handler = ImbalanceHandler(random_state=RANDOM_STATE)

resampled_data = {}

# 1. Baseline (pas de rééchantillonnage)
resampled_data['Sans rééchantillonnage'] = (X_train, y_train)
print(f'[Baseline] Pas de rééchantillonnage : {len(y_train):,} samples')

# 2. SMOTE
print('\nApplication de SMOTE...')
t0 = time.time()
X_smote, y_smote = handler.apply_smote(X_train, y_train, sampling_strategy=0.5)
resampled_data['SMOTE'] = (X_smote, y_smote)
print(f'  Durée : {time.time()-t0:.1f}s | {len(y_smote):,} samples (fraude: {sum(y_smote==1):,})')

# 3. ADASYN
print('\nApplication de ADASYN...')
t0 = time.time()
X_adasyn, y_adasyn = handler.apply_adasyn(X_train, y_train, sampling_strategy=0.5)
resampled_data['ADASYN'] = (X_adasyn, y_adasyn)
print(f'  Durée : {time.time()-t0:.1f}s | {len(y_adasyn):,} samples (fraude: {sum(y_adasyn==1):,})')

# 4. Random Undersampling
print('\nApplication de Random Undersampling...')
t0 = time.time()
X_under, y_under = handler.apply_random_undersampling(X_train, y_train, sampling_strategy=0.5)
resampled_data['Undersampling'] = (X_under, y_under)
print(f'  Durée : {time.time()-t0:.1f}s | {len(y_under):,} samples (fraude: {sum(y_under==1):,})')

# 5. SMOTE + Tomek
print('\nApplication de SMOTE + Tomek Links...')
t0 = time.time()
X_smt, y_smt = handler.apply_smote_tomek(X_train, y_train, sampling_strategy=0.5)
resampled_data['SMOTE+Tomek'] = (X_smt, y_smt)
print(f'  Durée : {time.time()-t0:.1f}s | {len(y_smt):,} samples (fraude: {sum(y_smt==1):,})')

# Résumé
print('\n' + '='*60)
print('Résumé des techniques appliquées :')
print(f'{"Technique":<25} {"Total":>10} {"Légitimes":>12} {"Fraudes":>10}')
print('-' * 60)
for name, (X_r, y_r) in resampled_data.items():
    n = len(y_r)
    n_f = int(sum(y_r == 1))
    n_l = n - n_f
    print(f'{name:<25} {n:>10,} {n_l:>12,} {n_f:>10,}')

In [ ]:
# Comparaison visuelle des tailles d'échantillons
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

techniques = list(resampled_data.keys())
totals = [len(y_r) for _, (_, y_r) in resampled_data.items()]
fraud_counts = [int(sum(y_r == 1)) for _, (_, y_r) in resampled_data.items()]
legit_counts = [t - f for t, f in zip(totals, fraud_counts)]

x = np.arange(len(techniques))
width = 0.35

# Grouped bar chart
bars1 = axes[0].bar(x - width/2, legit_counts, width, label='Légitime', color='#2ecc71', edgecolor='black')
bars2 = axes[0].bar(x + width/2, fraud_counts, width, label='Fraude', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Technique')
axes[0].set_ylabel('Nombre de samples')
axes[0].set_title('Taille des Datasets Rééchantillonnés', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(techniques, rotation=30, ha='right')
axes[0].set_yscale('log')
axes[0].legend()

# Ratio fraude/total
ratios = [f / t * 100 for f, t in zip(fraud_counts, totals)]
colors_bar = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c', '#e74c3c']
bars3 = axes[1].bar(techniques, ratios, color=colors_bar[:len(techniques)], edgecolor='black')
axes[1].set_xlabel('Technique')
axes[1].set_ylabel('% de Fraudes')
axes[1].set_title('Proportion de Fraudes après Rééchantillonnage', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
for bar, ratio in zip(bars3, ratios):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{ratio:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
fig.savefig(os.path.join(COMP_DIR, 'resampling_comparison_sizes.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {os.path.join(COMP_DIR, "resampling_comparison_sizes.png")}')

## Évaluation avec XGBoost

Chaque technique de rééchantillonnage est évaluée en entraînant un classifieur **XGBoost**
avec les mêmes hyperparamètres, puis en mesurant les performances sur le jeu de test
(non rééchantillonné).

Métriques évaluées : AUC-ROC, AUPRC, F1-Score, Précision, Rappel, Spécificité.

In [ ]:
# Entraînement et évaluation avec XGBoost pour chaque technique
results = {}

# Paramètres XGBoost communs
xgb_params = XGBOOST_PARAMS.copy()
xgb_params.pop('use_label_encoder', None)  # Deprecated in recent versions

# Ajouter la technique cost-sensitive
all_techniques = dict(resampled_data)  # Copie

for name, (X_r, y_r) in all_techniques.items():
    print(f'\n{"="*60}')
    print(f'Entraînement XGBoost — {name}')
    print(f'  Dataset : {len(y_r):,} samples ({int(sum(y_r==1)):,} fraudes)')

    t0 = time.time()
    model = XGBClassifier(**xgb_params)
    model.fit(X_r, y_r, verbose=False)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = compute_all_metrics(y_test, y_pred, y_proba)
    metrics['train_time_s'] = round(train_time, 2)
    results[name] = metrics

    print(f'  Durée entraînement : {train_time:.1f}s')
    print(f'  AUC-ROC: {metrics["auc_roc"]:.4f} | AUPRC: {metrics["auprc"]:.4f} | '
          f'F1: {metrics["f1_score"]:.4f} | Recall: {metrics["recall"]:.4f}')

# Cost-Sensitive (scale_pos_weight)
print(f'\n{"="*60}')
print('Entraînement XGBoost — Cost-Sensitive (scale_pos_weight)')
class_weights = ImbalanceHandler.get_class_weights(y_train)
scale_pos = class_weights[0] / class_weights[1] if class_weights[1] != 0 else 1
# scale_pos_weight = n_negative / n_positive
spw = int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1)

xgb_cost = XGBClassifier(**xgb_params, scale_pos_weight=spw)
t0 = time.time()
xgb_cost.fit(X_train, y_train, verbose=False)
train_time = time.time() - t0

y_pred_cost = xgb_cost.predict(X_test)
y_proba_cost = xgb_cost.predict_proba(X_test)[:, 1]

metrics_cost = compute_all_metrics(y_test, y_pred_cost, y_proba_cost)
metrics_cost['train_time_s'] = round(train_time, 2)
results['Cost-Sensitive'] = metrics_cost

print(f'  scale_pos_weight = {spw:.1f}')
print(f'  Durée entraînement : {train_time:.1f}s')
print(f'  AUC-ROC: {metrics_cost["auc_roc"]:.4f} | AUPRC: {metrics_cost["auprc"]:.4f} | '
      f'F1: {metrics_cost["f1_score"]:.4f} | Recall: {metrics_cost["recall"]:.4f}')

print(f'\n{"="*60}')
print('Évaluation terminée pour toutes les techniques.')

In [ ]:
# Tableau comparatif des métriques
metrics_to_show = ['auc_roc', 'auprc', 'f1_score', 'precision', 'recall',
                    'specificity', 'false_positive_rate', 'train_time_s']

comparison_df = pd.DataFrame(
    {tech: {m: results[tech][m] for m in metrics_to_show}
     for tech in results}
).T

comparison_df.columns = ['AUC-ROC', 'AUPRC', 'F1-Score', 'Précision',
                          'Rappel', 'Spécificité', 'Taux FP', 'Temps (s)']

# Trier par F1-Score décroissant
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print('Comparaison des Techniques de Rééchantillonnage :')
print('='*80)
display(comparison_df.round(4).style.highlight_max(
    subset=['AUC-ROC', 'AUPRC', 'F1-Score', 'Précision', 'Rappel', 'Spécificité'],
    color='lightgreen'
).highlight_min(
    subset=['Taux FP', 'Temps (s)'],
    color='lightgreen'
))

In [ ]:
# Visualisation comparative des métriques
metrics_plot = ['AUC-ROC', 'AUPRC', 'F1-Score', 'Précision', 'Rappel', 'Spécificité']
plot_df = comparison_df[metrics_plot]

fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(metrics_plot))
n_techniques = len(plot_df)
width = 0.8 / n_techniques

colors_palette = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c', '#e74c3c', '#f39c12']

for i, (technique, row) in enumerate(plot_df.iterrows()):
    offset = (i - n_techniques / 2 + 0.5) * width
    bars = ax.bar(x + offset, row.values, width,
                  label=technique, color=colors_palette[i % len(colors_palette)],
                  edgecolor='black', alpha=0.85)

ax.set_xlabel('Métrique', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Comparaison des Techniques de Rééchantillonnage — XGBoost',
             fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_plot, fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(COMP_DIR, 'imbalance_comparison.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {os.path.join(COMP_DIR, "imbalance_comparison.png")}')

## Analyse

### Synthèse des résultats

**SMOTE** offre le meilleur compromis entre les différentes métriques :
- Il augmente significativement le rappel (détection des fraudes) tout en maintenant
  une précision raisonnable, ce qui se traduit par le meilleur F1-Score.
- L'AUC-ROC et l'AUPRC restent élevés, confirmant une bonne capacité de discrimination
  sur l'ensemble du spectre de seuils.

**ADASYN** produit des résultats proches de SMOTE, avec une légère variation due à
sa nature adaptative qui génère davantage d'exemples synthétiques dans les zones
difficiles de l'espace des features.

**Random Undersampling** souffre d'une perte d'information significative :
- En supprimant une grande proportion de transactions légitimes, le modèle perd
  la capacité de distinguer finement les vrais négatifs.
- Cela entraîne un taux de faux positifs plus élevé et une spécificité réduite.
- Le rappel est souvent élevé, mais au prix d'une précision dégradée.

**SMOTE + Tomek Links** combine les avantages de SMOTE avec un nettoyage des
frontières de décision via les liens de Tomek. Les résultats sont similaires à
SMOTE pur, avec potentiellement des frontières mieux définies.

**Cost-Sensitive Learning** (via `scale_pos_weight`) est une alternative viable
qui ne modifie pas les données elles-mêmes :
- Aucun risque d'overfitting sur des exemples synthétiques
- Temps d'entraînement identique au baseline
- Bon compromis entre rappel et précision
- Recommandé comme approche complémentaire ou de référence

### Recommandation

Pour la suite du projet, nous retiendrons **SMOTE** comme technique principale de
rééchantillonnage, avec **Cost-Sensitive Learning** comme alternative. Les deux
approches seront testées avec chaque famille de modèles pour identifier la meilleure
combinaison (technique de rééchantillonnage x algorithme de classification).